# Atomic-GDN2: rounding, cancellation and AdamW — hypothesis test

**Akseleu Omirbay.** Run in a **fresh CPU Colab runtime after the TPU timing experiment has finished**. No TPU, tokens or Drive required. Run all cells; return the downloaded ZIP.

## Проверяемая гипотеза
Разный порядок вычисления одной рекурренции (OLD associative scan и JAX_REF chunked-WY) может давать разный знак очень малого FP32 градиента; AdamW может усиливать различие около epsilon. Результат не предрешён: отсутствие переворота знака тоже сохраняется.

## Дизайн
Три заранее заданных сида (0,1,2), B=1, L=256, H=1, D=4. Одни и те же сохранённые FP32 входы повышаются до FP64 без изменения значений. OLD, JAX_REF/WY и независимая последовательная рекурренция сравниваются в FP32 и **настоящем CPU FP64**. Независимый Python Decimal oracle использует 80 и 110 десятичных знаков, прямое дифференцирование рекурренции и контроль центральной конечной разностью.

Две цели: обычная квадратичная и **намеренно сконструированный cancellation stress case**. Во второй одна FP32 константа выбирается по Decimal oracle для почти нулевой суммы производных. Это демонстрация механизма, не частоты таких событий в обучении. Все методы получают одинаковую цель; подбора примера под конкретный алгоритм нет.

Также воспроизводится первый AdamW update из реально наблюдавшихся clipped gradients проблемной координаты. Это подтверждает усиление, но не устанавливает причину upstream ошибки. Отдельный нулевой контроль проверяет, что epsilon сам не создаёт adaptive update при нулевых градиенте и моментах.

## Границы вывода
Это **не запуск Pallas на CPU**. Сравнение OLD/WY изолирует смену алгоритма; связь NEW с JAX_REF проверяется отдельным TPU экспериментом. Уменьшение ошибки в FP64 относительно Decimal поддерживает объяснение округлением на этих входах. Не доказывает, что конкретный полный LM градиент математически равен нулю, что в ядре невозможны ошибки или что качество обучения не меняется.

Первая bias-corrected adaptive часть AdamW: `-lr*g/(abs(g)+eps)`; фактический replay использует Optax и weight decay. Предыдущие моменты здесь нулевые.

## Пакет для публикации
Последняя ячейка скачивает один ZIP: все входные NPZ, исходники с SHA256 и описанием изменений, pip freeze, сырые JSON и лог выполнения. В `publication/`: полные CSV таблицы (включая epsilon sweep), LaTeX таблица, график PDF/SVG/PNG, подпись, методы и ограничения, сводка проверок и execution attestation. Это запись воспроизводимого вычисления, а не независимая сертификация. Неполный или ошибочный запуск сохраняет свой статус; его нельзя выдавать за подтверждение.


In [ ]:
import datetime,hashlib,json,os,pathlib,subprocess,sys,zipfile
RUN_ID=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
WORK=pathlib.Path('/content')/('atomic-rounding-'+RUN_ID);WORK.mkdir()
OUT=WORK/'results';OUT.mkdir();SRC=OUT/'executed-source';SRC.mkdir()
subprocess.check_call([sys.executable,'-m','pip','install','-q','virtualenv'])
subprocess.check_call([sys.executable,'-m','virtualenv',str(WORK/'venv')])
PY=str(WORK/'venv/bin/python')
PACKAGES=['jax==0.11.2','jaxlib==0.11.2','numpy==2.1.3','scipy==1.16.3','optax==0.2.8','ml_dtypes==0.6.0','opt_einsum==3.4.0']
subprocess.check_call([PY,'-m','pip','install',*PACKAGES,'matplotlib'])
(OUT/'pip-freeze.txt').write_text(subprocess.check_output([PY,'-m','pip','freeze'],text=True))


In [ ]:
PAYLOAD={'hypothesis_old.py': "import jax\nimport jax.numpy as jnp\nPREC=jax.lax.Precision.HIGHEST\ndef old_impl(q,k,v,w,b,g,h0,scale):\n B,L,H,D=q.shape;C=128;N=L//C\n assert L%C==0\n alpha=jnp.exp(g.astype(q.dtype));ea=(b*k)*alpha;z=w*v\n def chunk(t):return jnp.moveaxis(t.reshape(B,N,C,H,D),1,0)\n def combine(s1,s2):\n  m1,c1=s1;m2,c2=s2\n  return jnp.matmul(m2,m1,precision=PREC),jnp.matmul(m2,c1,precision=PREC)+c2\n def step(carry,ci):\n  cM,cS=carry;kc,eac,zc,ac,qc=ci\n  M=jnp.eye(D,dtype=q.dtype)[None,None,None,:,:]*ac[...,None,:]-kc[..., :,None]*eac[...,None,:]\n  U=kc[..., :,None]*zc[...,None,:]\n  P,S=jax.lax.associative_scan(combine,(M,U),axis=1)\n  gM=jnp.einsum('bchmn,bhnp->bchmp',P,cM,precision=PREC)\n  gS=jnp.einsum('bchmn,bhnp->bchmp',P,cS,precision=PREC)+S\n  out=scale*jnp.einsum('bchij,bchi->bchj',gS,qc,precision=PREC)\n  return (gM[:,-1],gS[:,-1]),out\n eye=jnp.broadcast_to(jnp.eye(D,dtype=q.dtype),(B,H,D,D))\n (_,hf),out=jax.lax.scan(jax.checkpoint(step),(eye,h0),tuple(map(chunk,(k,ea,z,alpha,q))))\n return jnp.moveaxis(out,0,1).reshape(B,L,H,D),hf\n\ndef old_factory(scale):\n def impl(*args):return old_impl(*args,scale)\n @jax.custom_vjp\n def fn(*args):return impl(*args)\n def fwd(*args):\n  out,pullback=jax.vjp(jax.checkpoint(impl),*args)\n  return out,pullback\n def bwd(pullback,cotangent):return pullback(cotangent)\n fn.defvjp(fwd,bwd)\n return fn", 'rounding_adam_hypothesis.py': '"""Controlled CPU experiment: scan/WY rounding vs independent Decimal oracle and AdamW."""\nimport argparse,hashlib,importlib.metadata,json,math,os,platform,sys,time\nfrom decimal import Decimal,localcontext\nfrom pathlib import Path\nos.environ[\'JAX_PLATFORMS\']=\'cpu\'\nimport jax\nimport jax.numpy as jnp\nimport numpy as np\nimport optax\njax.config.update(\'jax_enable_x64\',True)\njax.config.update(\'jax_default_matmul_precision\',\'highest\')\nimport hypothesis_old as old\nimport hypothesis_wy32 as wy32\nimport hypothesis_wy64 as wy64\n\ndef exact(x):return Decimal.from_float(float(x))\ndef decimal_run(xs,direction,theta=Decimal(0),precision=80):\n """Independent literal recurrence plus forward tangent; no scan/WY implementation reuse."""\n with localcontext() as ctx:\n  ctx.prec=precision\n  q,k,v,w,b,g,h0=[np.asarray(x) for x in xs];D=q.shape[-1];L=q.shape[1]\n  state=[[exact(h0[0,0,i,j]) for j in range(D)] for i in range(D)]\n  tangent=[[Decimal(0) for j in range(D)] for i in range(D)]\n  outputs=[];derivatives=[]\n  for t in range(L):\n   qq,kk,vv,ww,bb,gg=[[exact(x[0,t,0,i]) for i in range(D)] for x in (q,k,v,w,b,g)]\n   dr=[exact(direction[0,t,0,i]) for i in range(D)]\n   alpha=[x.exp() for x in gg]\n   hs=[[alpha[i]*state[i][j] for j in range(D)] for i in range(D)]\n   ds=[[alpha[i]*tangent[i][j] for j in range(D)] for i in range(D)]\n   nv=[ww[j]*(vv[j]+theta*dr[j])-sum((bb[i]*kk[i]*hs[i][j] for i in range(D)),Decimal(0)) for j in range(D)]\n   nd=[ww[j]*dr[j]-sum((bb[i]*kk[i]*ds[i][j] for i in range(D)),Decimal(0)) for j in range(D)]\n   state=[[hs[i][j]+kk[i]*nv[j] for j in range(D)] for i in range(D)]\n   tangent=[[ds[i][j]+kk[i]*nd[j] for j in range(D)] for i in range(D)]\n   outputs.extend(Decimal(\'0.5\')*sum((qq[i]*state[i][j] for i in range(D)),Decimal(0)) for j in range(D))\n   derivatives.extend(Decimal(\'0.5\')*sum((qq[i]*tangent[i][j] for i in range(D)),Decimal(0)) for j in range(D))\n  return outputs+[x for row in state for x in row],derivatives+[x for row in tangent for x in row]\n\ndef decimal_objective(y,dy,coeff,mode,precision):\n with localcontext() as ctx:\n  ctx.prec=precision\n  if mode==\'natural\':\n   value=sum((x*x/2 for x in y),Decimal(0));terms=[a*b for a,b in zip(y,dy)]\n  else:\n   cs=[exact(x) for x in coeff];value=sum((a*b for a,b in zip(cs,y)),Decimal(0));terms=[a*b for a,b in zip(cs,dy)]\n  grad=sum(terms,Decimal(0));magnitude=sum(map(abs,terms),Decimal(0))\n  return value,grad,magnitude\n\ndef serial(q,k,v,w,b,g,h0):\n def step(h,inputs):\n  qt,kt,vt,wt,bt,gt=inputs\n  h=h*jnp.exp(gt)[..., :,None]\n  erase=jnp.einsum(\'bhd,bhdv->bhv\',bt*kt,h,precision=jax.lax.Precision.HIGHEST)\n  h=h+jnp.einsum(\'bhd,bhv->bhdv\',kt,wt*vt-erase,precision=jax.lax.Precision.HIGHEST)\n  o=.5*jnp.einsum(\'bhd,bhdv->bhv\',qt,h,precision=jax.lax.Precision.HIGHEST)\n  return h,o\n hf,o=jax.lax.scan(step,h0,tuple(jnp.moveaxis(x,1,0) for x in (q,k,v,w,b,g)))\n return jnp.moveaxis(o,0,1),hf\n\ndef inputs(seed,L,D):\n rng=np.random.default_rng(seed);shape=(1,L,1,D)\n q=rng.normal(size=shape).astype(np.float32);k=rng.normal(size=shape).astype(np.float32)\n q/=np.linalg.norm(q,axis=-1,keepdims=True);k/=np.linalg.norm(k,axis=-1,keepdims=True)\n v=(rng.normal(size=shape)*.2).astype(np.float32)\n w=rng.uniform(.2,.8,shape).astype(np.float32);b=rng.uniform(.2,.8,shape).astype(np.float32)\n g=-rng.uniform(.01,.04,shape).astype(np.float32)\n h=(rng.normal(size=(1,1,D,D))*.1).astype(np.float32)\n direction=(rng.normal(size=shape)*.2).astype(np.float32)\n return (q,k,v,w,b,g,h),direction,rng\n\ndef main(a):\n assert jax.default_backend()==\'cpu\' and jax.config.jax_enable_x64\n assert a.length%128==0\n report={\'status\':\'running\',\'scope\':\'Controlled reduced CPU recurrence, not the actual full-LM exact-gradient oracle\',\n  \'environment\':{\'backend\':jax.default_backend(),\'jax\':jax.__version__,\'numpy\':np.__version__,\'optax\':importlib.metadata.version(\'optax\'),\'x64\':True,\n    \'python\':sys.version,\'platform\':platform.platform(),\'devices\':[str(d) for d in jax.devices()],\n    \'default_matmul_precision\':str(jax.config.jax_default_matmul_precision)},\n  \'shape\':[1,a.length,1,a.dim],\'dtype_of_stored_inputs\':\'float32, promoted without changing their values\',\n  \'seeds_predeclared\':a.seeds,\'decimal_precision\':a.precision,\'cases\':[]}\n functions={}\n for dtype_name,dtype in [(\'fp32\',jnp.float32),(\'fp64\',jnp.float64)]:\n  for name in (\'SERIAL\',\'OLD\',\'WY\'):\n   fn=serial if name==\'SERIAL\' else old.old_factory(.5) if name==\'OLD\' else (wy32 if dtype_name==\'fp32\' else wy64).gdn2_chunked_wy_reference\n   for mode in (\'natural\',\'cancellation_stress\'):\n    def objective(theta,xs,direction,coeff,_name=name,_fn=fn,_mode=mode):\n     q,k,v,w,b,g,h=xs;vv=v+theta*direction\n     o,hf=_fn(q,k,vv,g,b,w,.5,128,h,wy_eps=0.) if _name==\'WY\' else _fn(q,k,vv,w,b,g,h)\n     y=jnp.concatenate([o.reshape(-1),hf.reshape(-1)])\n     return .5*jnp.sum(y*y) if _mode==\'natural\' else jnp.sum(y*coeff)\n    functions[(dtype_name,name,mode)]=jax.jit(jax.value_and_grad(objective))\n for seed in a.seeds:\n  print(\'START_SEED\',seed,\'Decimal precision\',a.precision,\'and\',a.precision+30,flush=True)\n  print(\'START_SEED\',seed,flush=True)\n  xs,dr,rng=inputs(seed,a.length,a.dim)\n  y,dy=decimal_run(xs,dr,precision=a.precision)\n  # Deliberate stress objective: choose a coefficient to cancel the high-precision\n  # tangent sum, THEN round/store that coefficient in fp32. No algorithm-dependent search.\n  coeff=rng.normal(0,.2,len(y)).astype(np.float32);pivot=max(range(len(dy)),key=lambda i:abs(dy[i]))\n  with localcontext() as ctx:\n   ctx.prec=a.precision\n   other=sum((exact(coeff[i])*dy[i] for i in range(len(dy)) if i!=pivot),Decimal(0))\n   coeff[pivot]=np.float32(float(-other/dy[pivot]))\n  assert all(np.isfinite(x).all() for x in xs) and np.isfinite(coeff).all()\n  digest=hashlib.sha256(b\'\'.join(x.tobytes() for x in (*xs,dr,coeff))).hexdigest()\n  np.savez_compressed(a.output.parent/f\'inputs-seed{seed}.npz\',**dict(zip((\'q\',\'k\',\'v\',\'w\',\'b\',\'g\',\'h0\'),xs)),direction=dr,coeff=coeff)\n  yhi,dyhi=decimal_run(xs,dr,precision=a.precision+30)\n  for mode in (\'natural\',\'cancellation_stress\'):\n   value,grad,terms=decimal_objective(y,dy,coeff,mode,a.precision)\n   vh,gh,th=decimal_objective(yhi,dyhi,coeff,mode,a.precision+30)\n   with localcontext() as ctx:\n    ctx.prec=a.precision\n    h=Decimal(\'1e-20\')\n    yp,dyp=decimal_run(xs,dr,theta=h,precision=a.precision)\n    ym,dym=decimal_run(xs,dr,theta=-h,precision=a.precision)\n    vp=decimal_objective(yp,dyp,coeff,mode,a.precision)[0]\n    vm=decimal_objective(ym,dym,coeff,mode,a.precision)[0]\n    fd=(vp-vm)/(2*h)\n    assert abs(fd-grad)<Decimal(\'1e-30\'),\'Independent tangent/finite-difference disagreement\'\n    assert abs(gh-grad)<Decimal(\'1e-30\'),\'Decimal oracle not precision-stable\'\n    condition=float(terms/abs(grad)) if grad else None\n   row={\'seed\':seed,\'mode\':mode,\'input_sha256\':digest,\'oracle_gradient_decimal\':str(gh),\n        \'oracle_value_decimal\':str(vh),\'oracle_precision_stability_abs\':float(abs(gh-grad)),\n        \'central_finite_difference_abs_error\':float(abs(fd-grad)),\n        \'derivative_sum_abs_terms\':float(terms),\'derivative_cancellation_ratio\':condition,\n        \'stress_construction\':\'Predeclared pivot=max(abs(Decimal tangent)); only one fp32 coefficient adjusted; same stored inputs/objective for all algorithms\',\n        \'oracle_first_Adam_adaptive_update_lr1e-3\':-1e-3*float(gh)/(abs(float(gh))+1e-8),\n        \'results\':[]}\n   for dtype_name,dtype in [(\'fp32\',jnp.float32),(\'fp64\',jnp.float64)]:\n    values=tuple(jnp.asarray(x,dtype) for x in xs);dd=jnp.asarray(dr,dtype);cc=jnp.asarray(coeff,dtype)\n    for name in (\'SERIAL\',\'OLD\',\'WY\'):\n     got,gg=functions[(dtype_name,name,mode)](jnp.asarray(0,dtype),values,dd,cc)\n     jax.block_until_ready((got,gg));g=float(gg)\n     assert math.isfinite(g) and math.isfinite(float(got))\n     with localcontext() as ctx:\n      ctx.prec=a.precision+30;err=abs(Decimal.from_float(g)-gh)\n     row[\'results\'].append({\'dtype\':dtype_name,\'algorithm\':name,\'gradient\':g,\'objective\':float(got),\n       \'gradient_abs_error_vs_decimal\':float(err),\'gradient_sign_matches_oracle\':bool((g>0)-(g<0)==(gh>0)-(gh<0)),\n       \'first_Adam_adaptive_update_lr1e-3\':-1e-3*g/(abs(g)+1e-8),\n       \'adaptive_update_abs_error_vs_decimal_gradient\':abs(-1e-3*g/(abs(g)+1e-8)-row[\'oracle_first_Adam_adaptive_update_lr1e-3\']),\n       \'epsilon_sensitivity\':{str(e):-1e-3*g/(abs(g)+e) for e in (1e-8,1e-7,1e-6)}})\n   row[\'fp32_OLD_WY_opposite_sign\']=bool(next(r[\'gradient\'] for r in row[\'results\'] if r[\'dtype\']==\'fp32\' and r[\'algorithm\']==\'OLD\')*next(r[\'gradient\'] for r in row[\'results\'] if r[\'dtype\']==\'fp32\' and r[\'algorithm\']==\'WY\')<0)\n   report[\'cases\'].append(row);save(a,report)\n   print(\'HYPOTHESIS_CASE\',seed,mode,\'oracle\',str(gh),\'cancellation_ratio\',condition,\'OLD_WY_sign_flip\',row[\'fp32_OLD_WY_opposite_sign\'],flush=True)\n # Verify actual Optax behavior: epsilon does not generate an adaptive update from zero.\n p=jnp.asarray([-.02789476327598095],jnp.float32);zero=jnp.zeros_like(p)\n tx=optax.adamw(1e-3,weight_decay=0.);u,_=tx.update(zero,tx.init(p),p)\n assert float(u[0])==0.\n report[\'zero_gradient_control\']={\'adaptive_update_with_zero_gradient_zero_moments_no_weight_decay\':float(u[0]),\n   \'epsilon_does_not_create_update\':True,\'with_weight_decay_only_expected\':float(-1e-3*.01*p[0])}\n # Replay actual observed clipped gradients; this establishes amplification only, not their exact upstream error.\n report[\'actual_coordinate_replay_provenance\']={\'coordinate\':\'block_6/q_conv_w[236,2]\',\n  \'seed\':0,\'previous_TPU_diagnostic_archive_sha256\':\'e3b05976c5f43ec09d59b940fbc6f5a1e603f1b8ae978c425f0f1e9121854d65\',\n  \'initial_parameter\':float(p[0]),\'lr\':.001,\'weight_decay\':.01,\'epsilon\':1e-8,\n  \'scope\':\'Previously observed clipped gradients transcribed into CPU replay, not a fresh full-LM execution.\'}\n report[\'actual_coordinate_replay\']=[]\n for name,g in [(\'NEW\',1.3835218731555054e-8),(\'OLD\',-9.705033074134847e-9)]:\n  tx=optax.adamw(1e-3,weight_decay=.01);u,_=tx.update(jnp.asarray([g],jnp.float32),tx.init(p),p)\n  report[\'actual_coordinate_replay\'].append({\'path\':name,\'observed_clipped_gradient\':g,\'Optax_first_update_CPU\':float(u[0])})\n report[\'status\']=\'completed\'\n report[\'interpretation\']={\n  \'conclusions_to_check\':[\'Do FP64 scan/WY errors shrink relative to the independent oracle?\',\n   \'Does cancellation stress increase sensitivity versus the natural objective?\',\n   \'Do float32 gradient differences amplify under first-step AdamW near epsilon?\'],\n  \'not_proven\':[\'The actual full-LM coordinate has mathematical gradient exactly zero\',\n   \'The original full-LM discrepancy is exclusively rounding and no implementation bug exists\',\n   \'Four completed training runs prove no effect on model quality\'],\n  \'scope\':\'Reduced stress case deliberately constructed for cancellation; not an estimate of its frequency in training.\'}\n save(a,report);print(\'HYPOTHESIS_EXPERIMENT_COMPLETED\',flush=True)\n\ndef save(a,report):\n temp=a.output.with_suffix(\'.tmp\');temp.write_text(json.dumps(report,indent=2,allow_nan=False));temp.replace(a.output)\nif __name__==\'__main__\':\n p=argparse.ArgumentParser();p.add_argument(\'--seeds\',nargs=\'+\',type=int,default=[0,1,2]);p.add_argument(\'--length\',type=int,default=256);p.add_argument(\'--dim\',type=int,default=4)\n p.add_argument(\'--precision\',type=int,default=80);p.add_argument(\'--output\',type=Path,required=True);a=p.parse_args();main(a)\n', 'rounding_publication_bundle.py': '"""Generate publication evidence, with explicit scope, from every completed case."""\nimport csv,hashlib,json,sys\nfrom pathlib import Path\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\n\ndef main(root):\n report=json.loads((root/\'report.json\').read_text());out=root/\'publication\';out.mkdir(exist_ok=True)\n rows=[];oracles=[]\n for case in report[\'cases\']:\n  oracles.append({k:case[k] for k in (\'seed\',\'mode\',\'oracle_gradient_decimal\',\'oracle_precision_stability_abs\',\'central_finite_difference_abs_error\',\'derivative_cancellation_ratio\',\'fp32_OLD_WY_opposite_sign\')})\n  for r in case[\'results\']:\n   rows.append(dict(seed=case[\'seed\'],mode=case[\'mode\'],dtype=r[\'dtype\'],algorithm=r[\'algorithm\'],gradient=r[\'gradient\'],absolute_error=r[\'gradient_abs_error_vs_decimal\'],sign_matches_oracle=r[\'gradient_sign_matches_oracle\'],adaptive_update=r[\'first_Adam_adaptive_update_lr1e-3\'],oracle_gradient=case[\'oracle_gradient_decimal\'],oracle_adaptive_update=case[\'oracle_first_Adam_adaptive_update_lr1e-3\'],input_sha256=case[\'input_sha256\']))\n def csvfile(name,data):\n  if data:\n   with (out/name).open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n    w=csv.DictWriter(f,fieldnames=list(data[0]));w.writeheader();w.writerows(data)\n csvfile(\'all-gradients-and-updates.csv\',rows);csvfile(\'oracle-checks.csv\',oracles)\n csvfile(\'observed-coordinate-adam-replay.csv\',report.get(\'actual_coordinate_replay\',[]))\n csvfile(\'epsilon-sensitivity.csv\',[dict(seed=c[\'seed\'],mode=c[\'mode\'],algorithm=r[\'algorithm\'],dtype=r[\'dtype\'],epsilon=e,adaptive_update=u) for c in report[\'cases\'] for r in c[\'results\'] for e,u in r[\'epsilon_sensitivity\'].items()])\n improvements=[]\n for case in report[\'cases\']:\n  for name in (\'SERIAL\',\'OLD\',\'WY\'):\n   pair={r[\'dtype\']:r for r in case[\'results\'] if r[\'algorithm\']==name}\n   e32=pair[\'fp32\'][\'gradient_abs_error_vs_decimal\'];e64=pair[\'fp64\'][\'gradient_abs_error_vs_decimal\']\n   improvements.append(dict(seed=case[\'seed\'],mode=case[\'mode\'],algorithm=name,fp32_error=e32,fp64_error=e64,fp64_error_smaller=e64<e32,error_reduction_factor=e32/e64 if e64 else None))\n csvfile(\'precision-convergence.csv\',improvements)\n stress=[c for c in report[\'cases\'] if c[\'mode\']==\'cancellation_stress\']\n natural=[c for c in report[\'cases\'] if c[\'mode\']==\'natural\']\n summary={\'experiment_status\':report[\'status\'],\'all_predeclared_cases_complete\':len(report[\'cases\'])==2*len(report[\'seeds_predeclared\']),\n \'controlled_stress_cases\':len(stress),\'controlled_stress_OLD_WY_fp32_opposite_signs\':sum(c[\'fp32_OLD_WY_opposite_sign\'] for c in stress),\n \'natural_cases\':len(natural),\'natural_OLD_WY_fp32_opposite_signs\':sum(c[\'fp32_OLD_WY_opposite_sign\'] for c in natural),\n \'precision_comparisons\':len(improvements),\'fp64_error_smaller_count\':sum(r[\'fp64_error_smaller\'] for r in improvements),\n \'maximum_decimal_precision_check_absolute_error\':max((c[\'oracle_precision_stability_abs\'] for c in report[\'cases\']),default=None),\n \'maximum_finite_difference_absolute_error\':max((c[\'central_finite_difference_abs_error\'] for c in report[\'cases\']),default=None),\n \'scope\':\'Controlled reduced CPU experiment. No full-LM exact-gradient or quality claim; no speed claim.\'}\n (out/\'evidence-summary.json\').write_text(json.dumps(summary,indent=2))\n (out/\'execution-attestation.json\').write_text(json.dumps({\n  \'author\':\'Akseleu Omirbay\',\'kind\':\'Self-recorded computational evidence; not independent certification\',\n  \'experiment_completed\':report[\'status\']==\'completed\' and summary[\'all_predeclared_cases_complete\'],\n  \'environment\':report[\'environment\'],\'stored_input_shape\':report[\'shape\'],\'seeds\':report[\'seeds_predeclared\'],\n  \'decimal_precision_digits\':[report[\'decimal_precision\'],report[\'decimal_precision\']+30],\n  \'report_sha256\':hashlib.sha256((root/\'report.json\').read_bytes()).hexdigest(),\n  \'observed_coordinate_replay_provenance\':report.get(\'actual_coordinate_replay_provenance\'),\n  \'scope\':summary[\'scope\']},indent=2))\n tex=[r\'\\begin{tabular}{lllr rr}\',r\'\\hline\',r\'Seed & Objective & Method & Precision & Gradient & Absolute error \\\\\',r\'\\hline\']\n for r in rows:\n  mode=\'Natural\' if r[\'mode\']==\'natural\' else \'Stress\'\n  tex.append(f"{r[\'seed\']} & {mode} & {r[\'algorithm\']} & {r[\'dtype\'].upper()} & {r[\'gradient\']:.5e} & {r[\'absolute_error\']:.3e}"+r\' \\\\\')\n tex.extend([r\'\\hline\',r\'\\end{tabular}\'])\n (out/\'all-cases-table.tex\').write_text(\'\\n\'.join(tex))\n plt.rcParams.update({\'font.family\':\'DejaVu Sans\',\'font.size\':9,\'axes.spines.top\':False,\'axes.spines.right\':False,\'pdf.fonttype\':42,\'ps.fonttype\':42})\n if stress:\n  fig,axes=plt.subplots(2,len(stress),figsize=(3.4*len(stress),5.5),squeeze=False)\n  colors={\'SERIAL\':\'#555555\',\'OLD\':\'#2166ac\',\'WY\':\'#b2182b\'}\n  for i,case in enumerate(stress):\n   labels=[]\n   for x,r in enumerate(case[\'results\']):\n    labels.append(r[\'algorithm\']+\'\\n\'+r[\'dtype\'].upper())\n    axes[0,i].scatter(x,max(r[\'gradient_abs_error_vs_decimal\'],1e-300),color=colors[r[\'algorithm\']],marker=\'o\' if r[\'dtype\']==\'fp32\' else \'s\',s=36)\n    axes[1,i].scatter(x,r[\'first_Adam_adaptive_update_lr1e-3\'],color=colors[r[\'algorithm\']],marker=\'o\' if r[\'dtype\']==\'fp32\' else \'s\',s=36)\n   axes[0,i].set_yscale(\'log\');axes[0,i].set_title(f"Seed {case[\'seed\']}")\n   axes[1,i].axhline(case[\'oracle_first_Adam_adaptive_update_lr1e-3\'],color=\'black\',ls=\'--\',lw=1,label=\'Decimal-gradient update\')\n   axes[1,i].axhline(0,color=\'#cccccc\',lw=.7)\n   for ax in axes[:,i]:ax.set_xticks(range(6),labels,fontsize=7);ax.grid(axis=\'y\',alpha=.2)\n   axes[1,i].ticklabel_format(axis=\'y\',style=\'sci\',scilimits=(0,0))\n  axes[0,0].set_ylabel(\'Absolute gradient error vs Decimal\');axes[1,0].set_ylabel(\'First adaptive Adam update\')\n  axes[1,-1].legend(fontsize=7,loc=\'best\')\n  fig.suptitle(\'Controlled cancellation stress test (reduced CPU recurrence)\',fontsize=11)\n  fig.tight_layout()\n  for ext in (\'pdf\',\'svg\',\'png\'):fig.savefig(out/f\'rounding-adam-stress.{ext}\',dpi=220,bbox_inches=\'tight\')\n  plt.close(fig)\n caption=\'\'\'Controlled cancellation stress test on a reduced GDN2 recurrence (CPU, B=1, L=256, H=1, D=4; seeds 0, 1, 2). SERIAL, OLD associative scan and pure-JAX chunked-WY use identical stored binary32 inputs and objective coefficients. FP64 promotes these values without changing them. Top: absolute scalar directional-gradient error against an independent sequential Decimal oracle (80 digits, checked at 110 digits and by central finite differences). Bottom: first bias-corrected adaptive Adam update, -0.001*g/(abs(g)+1e-8), with zero initial moments; dashed line uses the Decimal gradient. This panel excludes weight decay and global clipping; the separate observed-coordinate replay uses already-clipped gradients and Optax AdamW with weight decay. Stress coefficients intentionally create cancellation and do not estimate its prevalence in training. No Pallas kernel is executed here. Every predeclared seed is shown; ordinary quadratic-objective controls are included in the tables.\'\'\'\n (out/\'figure-caption.txt\').write_text(caption,encoding=\'utf-8\')\n methods=\'\'\'# Numerical sensitivity: methods and scope\n\nThis artifact tests a mechanism, not a predetermined conclusion. Stored float32 inputs are compared without changing their values. OLD and WY are derived transparently from archived benchmark sources; source derivations and hashes accompany the report. FP64 runs on CPU, not TPU. The independent Decimal oracle implements the literal sequential recurrence and forward tangent, without reusing scan/WY code. Decimal precision refinement and central finite differences check the oracle.\n\nBoth an ordinary quadratic objective and an intentionally constructed near-cancelling linear objective are reported for every predeclared seed. One stress coefficient is chosen using the oracle tangent and rounded to float32 before all algorithms run. This construction establishes possibility, not event frequency. Absolute errors are primary near zero; sign disagreement alone is not an accuracy metric.\n\nThe optimizer illustration uses zero initial moments. Its analytic update uses bias-corrected moments, excludes clipping and decay, and varies epsilon. A separate Optax replay starts from the previously observed clipped gradients; it tests amplification only. Epsilon does not generate an adaptive update from an exactly zero gradient with zero moments. Weight decay can still update parameters.\n\n## Interpretation rule\n\nIf FP64 OLD/WY approach the independent oracle while FP32 differences persist, the data support finite-precision sensitivity on these inputs. If this does not happen, investigate algorithmic or implementation differences. Do not silently relabel a failed check as passed. Inspect evidence-summary.json, all cases and logs before writing a conclusion.\n\n## Limits for publication\n\nThis reduced experiment does not establish the mathematically exact gradient of the original LM coordinate, exclude every Pallas implementation defect, establish long-run optimizer equivalence, or demonstrate unchanged validation quality. Previously completed training runs provide separate evidence of practical training behavior. Combine this experiment with the independent TPU NEW/JAX_REF comparison, but label their scopes separately. A controlled demonstration and an observed full-model discrepancy must not be described as the same experiment.\n\nNo independent statistical sampling claim is made from three predetermined seeds. Report all per-seed values rather than a significance test. These numbers are numerical diagnostics, not speedup measurements.\n\n## Files\n\nCSV files contain all cases, oracle checks, precision convergence and the observed-coordinate replay. The LaTeX table includes every algorithm, precision, objective and seed. Vector PDF/SVG and raster PNG show all stress cases. Original input arrays, exact executed sources, environment freeze, raw report and archive manifest support reproduction. Incomplete execution remains explicitly incomplete.\n\'\'\'\n (out/\'METHODS-AND-LIMITATIONS.md\').write_text(methods,encoding=\'utf-8\')\n (out/\'artifact-hashes.json\').write_text(json.dumps({p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in out.iterdir() if p.is_file() and p.name!=\'artifact-hashes.json\'},indent=2))\n print(\'PUBLICATION_EVIDENCE\',json.dumps(summary),flush=True)\n\nif __name__==\'__main__\':main(Path(sys.argv[1]))\n', 'hypothesis_wy32.py': '"""\nPure-JAX reference implementations for GDN-2.\nToken-serial (ground truth) and chunked-WY (backward fallback / cross-check).\n"""\nfrom __future__ import annotations\n\nimport jax\nimport jax.numpy as jnp\n\nREF_DTYPE = jnp.float32\n\n_HIGHEST = jax.lax.Precision.HIGHEST\n\n\ndef _wy_inverse(Akk, eps: float = 0.0):\n    """Row-by-row WY inverse via lax.scan. eps>0 damps Akk by (1-eps)\n    before inversion -- same Tikhonov-style regularization as the Pallas\n    block solve in gdn2_fwd.py\'s _block_solve, kept here so the reference\n    path (used by gdn2_pallas_forward_trainable\'s fallback on CPU/GPU, and\n    as a cross-check target for the Pallas path) solves the SAME damped\n    problem instead of the exact one. Without this, gradient/output\n    comparisons between the two paths would diverge whenever a chunk\'s\n    Akk is near-singular -- exactly the case damping exists to handle.\n    Default eps=0.0 preserves the original exact behavior for anyone\n    relying on this function directly.\n    """\n    C = Akk.shape[-1]\n    eye = jnp.eye(C, dtype=Akk.dtype)\n    batch_shape = Akk.shape[:-2]\n\n    Akk_damped = Akk * (1.0 - eps)\n\n    def row_step(A_rows, i):\n        t_row = jnp.take(Akk_damped, i, axis=-2)\n        contrib = jnp.einsum("...j,...jk->...k", t_row, A_rows, precision=_HIGHEST)\n        new_row = eye[i] - contrib\n        A_rows = jax.lax.dynamic_update_slice_in_dim(A_rows, new_row[..., None, :], i, axis=-2)\n        return A_rows, None\n\n    A0 = jnp.zeros(batch_shape + (C, C), dtype=Akk.dtype)\n    A_final, _ = jax.lax.scan(row_step, A0, jnp.arange(C))\n    return A_final\n\n\ndef gdn2_token_serial_reference(q, k, v, g, b, w, scale, h0=None):\n    """Ground-truth token-by-token scan. Slow but numerically exact."""\n    bsz, L, H, D = q.shape\n    Dv = v.shape[-1]\n    dtype = q.dtype\n\n    alpha = jnp.exp(g.astype(REF_DTYPE))\n\n    if h0 is None:\n        h0 = jnp.zeros((bsz, H, D, Dv), dtype=REF_DTYPE)\n\n    def step(h, inputs):\n        q_t, k_t, v_t, alpha_t, b_t, w_t = inputs\n        h = h * alpha_t[..., :, None]\n        bk_t = (b_t * k_t).astype(REF_DTYPE)\n        erase = jnp.einsum("bhd,bhdv->bhv", bk_t, h, precision=_HIGHEST)\n        v_new = (w_t * v_t).astype(REF_DTYPE) - erase\n        h = h + jnp.einsum("bhd,bhv->bhdv", k_t.astype(REF_DTYPE), v_new, precision=_HIGHEST)\n        o_t = jnp.einsum("bhdv,bhd->bhv", h, (q_t * scale).astype(REF_DTYPE), precision=_HIGHEST)\n        return h, o_t\n\n    to_scan = tuple(jnp.moveaxis(x, 1, 0) for x in (q, k, v, alpha, b, w))\n    h_final, o_scanned = jax.lax.scan(step, h0, to_scan)\n    o = jnp.moveaxis(o_scanned, 0, 1).astype(dtype)\n    return o, h_final\n\n\ndef _build_chunk_wy(q_c, k_c, v_c, g_raw_c, b_c, w_c, scale, wy_eps: float = 0.0):\n    C = q_c.shape[1]\n    f32 = REF_DTYPE\n\n    gc = jnp.cumsum(g_raw_c.astype(f32), axis=1)\n    gc_bhcd = jnp.moveaxis(gc, 2, 1)\n    decay_diff = gc_bhcd[:, :, :, None, :] - gc_bhcd[:, :, None, :, :]\n    edecay = jnp.exp(jnp.clip(decay_diff, -20.0, 20.0))\n\n    causal = jnp.tril(jnp.ones((C, C), dtype=f32))\n    strict = jnp.tril(jnp.ones((C, C), dtype=f32), k=-1)\n\n    q_bhcd = jnp.moveaxis(q_c, 2, 1).astype(f32)\n    k_bhcd = jnp.moveaxis(k_c, 2, 1).astype(f32)\n    b_bhcd = jnp.moveaxis(b_c, 2, 1).astype(f32)\n\n    Aqk = scale * jnp.einsum("bhid,bhijd,bhjd->bhij", q_bhcd, edecay, k_bhcd, precision=_HIGHEST) * causal\n    bk_bhcd = b_bhcd * k_bhcd\n    Akk = jnp.einsum("bhid,bhijd,bhjd->bhij", bk_bhcd, edecay, k_bhcd, precision=_HIGHEST) * strict\n\n    Aqk = jnp.nan_to_num(Aqk, nan=0.0, posinf=1e4, neginf=-1e4)\n    Akk = jnp.nan_to_num(Akk, nan=0.0, posinf=1e4, neginf=-1e4)\n\n    A = _wy_inverse(Akk, eps=wy_eps)\n    A = jnp.nan_to_num(A, nan=0.0, posinf=1e4, neginf=-1e4)\n\n    kb_decayed = (b_c.astype(f32) * k_c.astype(f32)) * jnp.exp(gc)\n    w_pseudo = jnp.einsum("bhij,bjhd->bihd", A, kb_decayed, precision=_HIGHEST)\n    u = jnp.einsum("bhij,bjhv->bihv", A, (w_c * v_c).astype(f32), precision=_HIGHEST)\n    w_pseudo = jnp.nan_to_num(w_pseudo, nan=0.0, posinf=1e4, neginf=-1e4)\n    u = jnp.nan_to_num(u, nan=0.0, posinf=1e4, neginf=-1e4)\n\n    gc_last = gc[:, -1]\n    kg = k_c.astype(f32) * jnp.exp(gc_last[:, None] - gc)\n    qg = q_c.astype(f32) * jnp.exp(gc)\n\n    return Aqk, w_pseudo, u, kg, qg, gc_last\n\n\ndef gdn2_chunked_wy_reference(q, k, v, g, b, w, scale, chunk_size, h0=None, wy_eps: float = 0.0):\n    """Chunked-WY reference with jax.checkpoint for memory efficiency.\n\n    wy_eps: pass config.wy_eps here when using this as a cross-check\n    against the Pallas path (gdn2_pallas_forward_trainable), otherwise the\n    two paths solve slightly different problems on near-singular chunks\n    and comparisons/asserts between them will show spurious mismatches.\n    """\n    bsz, L, H, D = q.shape\n    Dv = v.shape[-1]\n    dtype = q.dtype\n    if L % chunk_size != 0:\n        raise ValueError(f"seq_len={L} must be divisible by chunk_size={chunk_size}")\n    n_chunks = L // chunk_size\n\n    def to_chunks(t):\n        shp = t.shape\n        t = t.reshape(bsz, n_chunks, chunk_size, *shp[2:])\n        return jnp.moveaxis(t, 1, 0)\n\n    q_ch, k_ch, v_ch, g_ch, b_ch, w_ch = map(to_chunks, (q, k, v, g, b, w))\n\n    if h0 is None:\n        h0 = jnp.zeros((bsz, H, D, Dv), dtype=REF_DTYPE)\n\n    def chunk_step(h_pre, inputs):\n        q_c, k_c, v_c, g_c, b_c, w_c = inputs\n        Aqk, w_pseudo, u, kg, qg, gc_last = _build_chunk_wy(q_c, k_c, v_c, g_c, b_c, w_c, scale, wy_eps=wy_eps)\n\n        wh = jnp.einsum("bihd,bhdv->bihv", w_pseudo, h_pre, precision=_HIGHEST)\n        v_new = u - wh\n\n        qh = jnp.einsum("bihd,bhdv->bihv", qg, h_pre, precision=_HIGHEST)\n        v_new_bhcv = jnp.moveaxis(v_new, 2, 1)\n        intra = jnp.einsum("bhij,bhjv->bhiv", Aqk, v_new_bhcv, precision=_HIGHEST)\n        intra = jnp.moveaxis(intra, 1, 2)\n        o_c = scale * qh + intra\n\n        decay_h = jnp.exp(gc_last)[..., None]\n        write = jnp.einsum("bihd,bihv->bhdv", kg, v_new, precision=_HIGHEST)\n        h_new = h_pre * decay_h + write\n        h_new = jnp.nan_to_num(jnp.clip(h_new, -1e4, 1e4), nan=0.0, posinf=1e4, neginf=-1e4)\n        o_c = jnp.nan_to_num(o_c, nan=0.0, posinf=1e4, neginf=-1e4)\n\n        return h_new, o_c\n\n    chunk_step = jax.checkpoint(chunk_step)\n\n    h_final, o_scanned = jax.lax.scan(chunk_step, h0, (q_ch, k_ch, v_ch, g_ch, b_ch, w_ch))\n    o = jnp.moveaxis(o_scanned, 0, 1).reshape(bsz, L, H, Dv)\n    return o, h_final\n', 'hypothesis_wy64.py': '"""\nPure-JAX reference implementations for GDN-2.\nToken-serial (ground truth) and chunked-WY (backward fallback / cross-check).\n"""\nfrom __future__ import annotations\n\nimport jax\nimport jax.numpy as jnp\n\nREF_DTYPE = jnp.float64\n\n_HIGHEST = jax.lax.Precision.HIGHEST\n\n\ndef _wy_inverse(Akk, eps: float = 0.0):\n    """Row-by-row WY inverse via lax.scan. eps>0 damps Akk by (1-eps)\n    before inversion -- same Tikhonov-style regularization as the Pallas\n    block solve in gdn2_fwd.py\'s _block_solve, kept here so the reference\n    path (used by gdn2_pallas_forward_trainable\'s fallback on CPU/GPU, and\n    as a cross-check target for the Pallas path) solves the SAME damped\n    problem instead of the exact one. Without this, gradient/output\n    comparisons between the two paths would diverge whenever a chunk\'s\n    Akk is near-singular -- exactly the case damping exists to handle.\n    Default eps=0.0 preserves the original exact behavior for anyone\n    relying on this function directly.\n    """\n    C = Akk.shape[-1]\n    eye = jnp.eye(C, dtype=Akk.dtype)\n    batch_shape = Akk.shape[:-2]\n\n    Akk_damped = Akk * (1.0 - eps)\n\n    def row_step(A_rows, i):\n        t_row = jnp.take(Akk_damped, i, axis=-2)\n        contrib = jnp.einsum("...j,...jk->...k", t_row, A_rows, precision=_HIGHEST)\n        new_row = eye[i] - contrib\n        A_rows = jax.lax.dynamic_update_slice_in_dim(A_rows, new_row[..., None, :], i, axis=-2)\n        return A_rows, None\n\n    A0 = jnp.zeros(batch_shape + (C, C), dtype=Akk.dtype)\n    A_final, _ = jax.lax.scan(row_step, A0, jnp.arange(C))\n    return A_final\n\n\ndef gdn2_token_serial_reference(q, k, v, g, b, w, scale, h0=None):\n    """Ground-truth token-by-token scan. Slow but numerically exact."""\n    bsz, L, H, D = q.shape\n    Dv = v.shape[-1]\n    dtype = q.dtype\n\n    alpha = jnp.exp(g.astype(REF_DTYPE))\n\n    if h0 is None:\n        h0 = jnp.zeros((bsz, H, D, Dv), dtype=REF_DTYPE)\n\n    def step(h, inputs):\n        q_t, k_t, v_t, alpha_t, b_t, w_t = inputs\n        h = h * alpha_t[..., :, None]\n        bk_t = (b_t * k_t).astype(REF_DTYPE)\n        erase = jnp.einsum("bhd,bhdv->bhv", bk_t, h, precision=_HIGHEST)\n        v_new = (w_t * v_t).astype(REF_DTYPE) - erase\n        h = h + jnp.einsum("bhd,bhv->bhdv", k_t.astype(REF_DTYPE), v_new, precision=_HIGHEST)\n        o_t = jnp.einsum("bhdv,bhd->bhv", h, (q_t * scale).astype(REF_DTYPE), precision=_HIGHEST)\n        return h, o_t\n\n    to_scan = tuple(jnp.moveaxis(x, 1, 0) for x in (q, k, v, alpha, b, w))\n    h_final, o_scanned = jax.lax.scan(step, h0, to_scan)\n    o = jnp.moveaxis(o_scanned, 0, 1).astype(dtype)\n    return o, h_final\n\n\ndef _build_chunk_wy(q_c, k_c, v_c, g_raw_c, b_c, w_c, scale, wy_eps: float = 0.0):\n    C = q_c.shape[1]\n    f32 = REF_DTYPE\n\n    gc = jnp.cumsum(g_raw_c.astype(f32), axis=1)\n    gc_bhcd = jnp.moveaxis(gc, 2, 1)\n    decay_diff = gc_bhcd[:, :, :, None, :] - gc_bhcd[:, :, None, :, :]\n    edecay = jnp.exp(jnp.clip(decay_diff, -20.0, 20.0))\n\n    causal = jnp.tril(jnp.ones((C, C), dtype=f32))\n    strict = jnp.tril(jnp.ones((C, C), dtype=f32), k=-1)\n\n    q_bhcd = jnp.moveaxis(q_c, 2, 1).astype(f32)\n    k_bhcd = jnp.moveaxis(k_c, 2, 1).astype(f32)\n    b_bhcd = jnp.moveaxis(b_c, 2, 1).astype(f32)\n\n    Aqk = scale * jnp.einsum("bhid,bhijd,bhjd->bhij", q_bhcd, edecay, k_bhcd, precision=_HIGHEST) * causal\n    bk_bhcd = b_bhcd * k_bhcd\n    Akk = jnp.einsum("bhid,bhijd,bhjd->bhij", bk_bhcd, edecay, k_bhcd, precision=_HIGHEST) * strict\n\n    Aqk = jnp.nan_to_num(Aqk, nan=0.0, posinf=1e4, neginf=-1e4)\n    Akk = jnp.nan_to_num(Akk, nan=0.0, posinf=1e4, neginf=-1e4)\n\n    A = _wy_inverse(Akk, eps=wy_eps)\n    A = jnp.nan_to_num(A, nan=0.0, posinf=1e4, neginf=-1e4)\n\n    kb_decayed = (b_c.astype(f32) * k_c.astype(f32)) * jnp.exp(gc)\n    w_pseudo = jnp.einsum("bhij,bjhd->bihd", A, kb_decayed, precision=_HIGHEST)\n    u = jnp.einsum("bhij,bjhv->bihv", A, (w_c * v_c).astype(f32), precision=_HIGHEST)\n    w_pseudo = jnp.nan_to_num(w_pseudo, nan=0.0, posinf=1e4, neginf=-1e4)\n    u = jnp.nan_to_num(u, nan=0.0, posinf=1e4, neginf=-1e4)\n\n    gc_last = gc[:, -1]\n    kg = k_c.astype(f32) * jnp.exp(gc_last[:, None] - gc)\n    qg = q_c.astype(f32) * jnp.exp(gc)\n\n    return Aqk, w_pseudo, u, kg, qg, gc_last\n\n\ndef gdn2_chunked_wy_reference(q, k, v, g, b, w, scale, chunk_size, h0=None, wy_eps: float = 0.0):\n    """Chunked-WY reference with jax.checkpoint for memory efficiency.\n\n    wy_eps: pass config.wy_eps here when using this as a cross-check\n    against the Pallas path (gdn2_pallas_forward_trainable), otherwise the\n    two paths solve slightly different problems on near-singular chunks\n    and comparisons/asserts between them will show spurious mismatches.\n    """\n    bsz, L, H, D = q.shape\n    Dv = v.shape[-1]\n    dtype = q.dtype\n    if L % chunk_size != 0:\n        raise ValueError(f"seq_len={L} must be divisible by chunk_size={chunk_size}")\n    n_chunks = L // chunk_size\n\n    def to_chunks(t):\n        shp = t.shape\n        t = t.reshape(bsz, n_chunks, chunk_size, *shp[2:])\n        return jnp.moveaxis(t, 1, 0)\n\n    q_ch, k_ch, v_ch, g_ch, b_ch, w_ch = map(to_chunks, (q, k, v, g, b, w))\n\n    if h0 is None:\n        h0 = jnp.zeros((bsz, H, D, Dv), dtype=REF_DTYPE)\n\n    def chunk_step(h_pre, inputs):\n        q_c, k_c, v_c, g_c, b_c, w_c = inputs\n        Aqk, w_pseudo, u, kg, qg, gc_last = _build_chunk_wy(q_c, k_c, v_c, g_c, b_c, w_c, scale, wy_eps=wy_eps)\n\n        wh = jnp.einsum("bihd,bhdv->bihv", w_pseudo, h_pre, precision=_HIGHEST)\n        v_new = u - wh\n\n        qh = jnp.einsum("bihd,bhdv->bihv", qg, h_pre, precision=_HIGHEST)\n        v_new_bhcv = jnp.moveaxis(v_new, 2, 1)\n        intra = jnp.einsum("bhij,bhjv->bhiv", Aqk, v_new_bhcv, precision=_HIGHEST)\n        intra = jnp.moveaxis(intra, 1, 2)\n        o_c = scale * qh + intra\n\n        decay_h = jnp.exp(gc_last)[..., None]\n        write = jnp.einsum("bihd,bihv->bhdv", kg, v_new, precision=_HIGHEST)\n        h_new = h_pre * decay_h + write\n        h_new = jnp.nan_to_num(jnp.clip(h_new, -1e4, 1e4), nan=0.0, posinf=1e4, neginf=-1e4)\n        o_c = jnp.nan_to_num(o_c, nan=0.0, posinf=1e4, neginf=-1e4)\n\n        return h_new, o_c\n\n    chunk_step = jax.checkpoint(chunk_step)\n\n    h_final, o_scanned = jax.lax.scan(chunk_step, h0, (q_ch, k_ch, v_ch, g_ch, b_ch, w_ch))\n    o = jnp.moveaxis(o_scanned, 0, 1).reshape(bsz, L, H, Dv)\n    return o, h_final\n'}
HASHES={'hypothesis_old.py': '243d1a0ee4096ff7e6220bc27150c6a589a71ef562394184dc3ffd4d773a5c4b', 'rounding_adam_hypothesis.py': 'aeb6af4ee43f9da520f20174f001187087a2770508081c5d33d841861cc4e7de', 'rounding_publication_bundle.py': 'e621ed2705ac75cd8366043000cc981fc352e799a1b87d49d6961dbf3c5c0c1d', 'hypothesis_wy32.py': '97fa8f28564d098c6200ff910f937e550328e0cc5b5d403d3155a50765b08a23', 'hypothesis_wy64.py': '8eb1a9523f21605d24c2a88541fe431e265e56f711cbe2888ebfd3580e7f5ae0'}
DERIVATION={'archive_sha256': '97de320f2d026dd0bb325f6347c92376ee848dfbd842985f423d069def4d7797', 'original_sources_sha256': {'candidate_common.py': '03a514716fb4e1279e1c9e96eb8f1458e3627ac86124250dd3c8af6512176d10', 'atomic_ops/reference.py': 'ee7ecc97a2ef474e29219256646ee7d118045cc798798aaed6851fd0d246ad3d'}, 'transformations': ['OLD: extract old_impl and old_factory; g.astype(float32) -> g.astype(q.dtype), identical for fp32.', 'WY: replace hardcoded jnp.float32 with REF_DTYPE; separate fp32 and fp64 modules. Algorithm unchanged.', 'Independent SERIAL and Decimal implementations written from literal recurrence; no WY/scan reuse.'], 'derived_sources_sha256': {'hypothesis_old.py': '243d1a0ee4096ff7e6220bc27150c6a589a71ef562394184dc3ffd4d773a5c4b', 'rounding_adam_hypothesis.py': 'aeb6af4ee43f9da520f20174f001187087a2770508081c5d33d841861cc4e7de', 'rounding_publication_bundle.py': 'e621ed2705ac75cd8366043000cc981fc352e799a1b87d49d6961dbf3c5c0c1d', 'hypothesis_wy32.py': '97fa8f28564d098c6200ff910f937e550328e0cc5b5d403d3155a50765b08a23', 'hypothesis_wy64.py': '8eb1a9523f21605d24c2a88541fe431e265e56f711cbe2888ebfd3580e7f5ae0'}}
for n,s in PAYLOAD.items():
 p=SRC/n;p.write_text(s,encoding='utf-8');assert hashlib.sha256(p.read_bytes()).hexdigest()==HASHES[n],n
(OUT/'source-derivation.json').write_text(json.dumps(DERIVATION,indent=2))
print('SOURCE_HASHES_VERIFIED')


In [ ]:
env=dict(os.environ,JAX_PLATFORMS='cpu',JAX_ENABLE_X64='true',PYTHONUNBUFFERED='1')
command=[PY,'-u',str(SRC/'rounding_adam_hypothesis.py'),'--output',str(OUT/'report.json')]
status={'command':command,'status':'running'}
try:
 with (OUT/'worker.log').open('w') as log:
  process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,env=env,bufsize=1)
  for line in process.stdout:print(line,end='',flush=True);log.write(line);log.flush()
  status.update(status='completed' if process.wait()==0 else 'error',exit_code=process.returncode)
except BaseException as e:
 status.update(status='interrupted_or_error',error=repr(e))
 if 'process' in globals() and process.poll() is None:process.terminate();process.wait()
finally:
 (OUT/'status.json').write_text(json.dumps(status,indent=2))
print(status)


In [ ]:
p=OUT/'report.json'
if p.exists():
 analysis_code=subprocess.call([PY,'-u',str(SRC/'rounding_publication_bundle.py'),str(OUT)],env=env)
 (OUT/'publication-generation-status.json').write_text(json.dumps({'exit_code':analysis_code}))
 r=json.loads(p.read_text())
 print('Experiment status:',r['status'])
 for case in r['cases']:
  print('SEED',case['seed'],case['mode'],'Decimal gradient',case['oracle_gradient_decimal'])
  for x in case['results']:print(x['dtype'],x['algorithm'],'g=',x['gradient'],'absolute error=',x['gradient_abs_error_vs_decimal'])
  print('FP32 OLD/WY opposite signs:',case['fp32_OLD_WY_opposite_sign'])
 print(r.get('interpretation',{}))
manifest={str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.rglob('*') if p.is_file()}
DEST=WORK.parent/('Atomic-gdn2-rounding-hypothesis-results-'+RUN_ID+'.zip')
with zipfile.ZipFile(DEST,'w',zipfile.ZIP_DEFLATED) as z:
 for n in manifest:z.write(OUT/n,n)
 z.writestr('manifest.json',json.dumps({'files':manifest},indent=2))
with zipfile.ZipFile(DEST) as z:assert all(hashlib.sha256(z.read(n)).hexdigest()==h for n,h in manifest.items())
print('ARCHIVE',DEST,'SHA256',hashlib.sha256(DEST.read_bytes()).hexdigest())
from google.colab import files
files.download(str(DEST))
